# QuickPay FinTech Pipeline
**Student Assignment — Parts 3, 4, and Dashboard Source Outputs**

This notebook covers:
- Part 3: Python Reconciliation Workflow (ledger.csv vs gateway.csv)
- Part 4: JSON Normalization (api_response_sample.json)
- Dashboard Source Outputs (daily_summary, payment_method_breakdown, region_breakdown, merchant_performance_summary)


## Setup — Imports and Paths

In [ ]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path

RAW = Path('../01_data/raw')
PROCESSED = Path('../01_data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Libraries loaded. Working directories set.")
print(f"  RAW: {RAW}")
print(f"  PROCESSED: {PROCESSED}")


Libraries loaded. Working directories set.
  RAW: ../01_data/raw
  PROCESSED: ../01_data/processed


---
## Part 3: Reconciliation Workflow

Reconcile internal ledger records against gateway settlement records.
This identifies financial discrepancies that need investigation or correction.


### Step 1 — Load Ledger and Gateway Files

In [ ]:
ledger = pd.read_csv(RAW / 'ledger.csv', dtype=str)
gateway = pd.read_csv(RAW / 'gateway.csv', dtype=str)

# Convert amounts to float
ledger['amount_usd'] = ledger['amount_usd'].astype(float)
gateway['amount_usd'] = gateway['amount_usd'].astype(float)

print("Ledger shape:", ledger.shape)
print("Gateway shape:", gateway.shape)
print("\nLedger columns:", ledger.columns.tolist())
print("Gateway columns:", gateway.columns.tolist())


Ledger shape: (10, 6)
Gateway shape: (9, 6)
Ledger columns: ['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd', 'status', 'payment_method']
Gateway columns: ['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd', 'status', 'payment_method']


### Step 2 — Quality Checks: Duplicates and Nulls

In [ ]:
print("=== LEDGER QC ===")
print(f"Total rows: {len(ledger)}")
print(f"Duplicate transaction_ids: {ledger['transaction_id'].duplicated().sum()}")
print(f"Null values per column:")
print(ledger.isnull().sum().to_string())

print("\n=== GATEWAY QC ===")
print(f"Total rows: {len(gateway)}")
print(f"Duplicate transaction_ids: {gateway['transaction_id'].duplicated().sum()}")
print(f"Null values per column:")
print(gateway.isnull().sum().to_string())

print("\n>>> No duplicates or nulls found in either file.")


=== LEDGER QC ===
Total rows: 10
Duplicate transaction_ids: 0
Null values per column:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

=== GATEWAY QC ===
Total rows: 9
Duplicate transaction_ids: 0
Null values per column:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

>>> No duplicates or nulls found in either file.


### Step 3 — Records Missing in Gateway (in ledger but not in gateway)

In [ ]:
ledger_ids = set(ledger['transaction_id'])
gateway_ids = set(gateway['transaction_id'])

missing_in_gateway = ledger[~ledger['transaction_id'].isin(gateway_ids)].copy()
print(f"Records in ledger but NOT in gateway: {len(missing_in_gateway)}")
print(missing_in_gateway.to_string(index=False))

missing_in_gateway.to_csv(PROCESSED / 'missing_in_gateway.csv', index=False)
print("\nSaved: missing_in_gateway.csv")


Records in ledger but NOT in gateway: 2
 transaction_id transaction_date merchant_id  amount_usd   status payment_method
           R004       2026-03-02        M003      2100.0  success           Card
           R010       2026-03-05        M004      2500.0  success         Wallet

Saved: missing_in_gateway.csv


### Step 4 — Records Missing in Ledger (in gateway but not in ledger)

In [ ]:
missing_in_ledger = gateway[~gateway['transaction_id'].isin(ledger_ids)].copy()
print(f"Records in gateway but NOT in ledger: {len(missing_in_ledger)}")
print(missing_in_ledger.to_string(index=False))

missing_in_ledger.to_csv(PROCESSED / 'missing_in_ledger.csv', index=False)
print("\nSaved: missing_in_ledger.csv")


Records in gateway but NOT in ledger: 1
 transaction_id transaction_date merchant_id  amount_usd   status payment_method
           R011       2026-03-05        M003      1800.0  success           Card

Saved: missing_in_ledger.csv


### Step 5 — Amount Mismatches (same transaction_id, different amount_usd)

In [ ]:
merged = ledger.merge(gateway, on='transaction_id', suffixes=('_ledger', '_gateway'))

amount_mismatches = merged[merged['amount_usd_ledger'] != merged['amount_usd_gateway']].copy()
amount_mismatches['amount_diff'] = (amount_mismatches['amount_usd_gateway'] - amount_mismatches['amount_usd_ledger']).round(2)

print(f"Amount mismatches found: {len(amount_mismatches)}")
cols = ['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway', 'amount_diff', 'status_ledger', 'status_gateway']
print(amount_mismatches[cols].to_string(index=False))

amount_mismatches[cols].to_csv(PROCESSED / 'amount_mismatches.csv', index=False)
print("\nSaved: amount_mismatches.csv")


Amount mismatches found: 2
 transaction_id  amount_usd_ledger  amount_usd_gateway  amount_diff status_ledger status_gateway
           R002              850.0               900.0         50.0       success        success
           R008              640.0               600.0        -40.0       success        success

Saved: amount_mismatches.csv


### Step 6 — Status Mismatches (same transaction_id, different status)

In [ ]:
status_mismatches = merged[merged['status_ledger'] != merged['status_gateway']].copy()
print(f"Status mismatches found: {len(status_mismatches)}")

s_cols = ['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway', 'status_ledger', 'status_gateway']
print(status_mismatches[s_cols].to_string(index=False))

status_mismatches[s_cols].to_csv(PROCESSED / 'status_mismatches.csv', index=False)
print("\nSaved: status_mismatches.csv")


Status mismatches found: 1
 transaction_id  amount_usd_ledger  amount_usd_gateway status_ledger status_gateway
           R005             7200.0              7200.0       success         failed

Saved: status_mismatches.csv


### Step 7 — Final Reconciliation Report

In [ ]:
recon = []

for _, r in missing_in_gateway.iterrows():
    recon.append({'transaction_id': r['transaction_id'], 'issue': 'missing_in_gateway',
                  'ledger_amount': r['amount_usd'], 'gateway_amount': np.nan,
                  'ledger_status': r['status'], 'gateway_status': np.nan})

for _, r in missing_in_ledger.iterrows():
    recon.append({'transaction_id': r['transaction_id'], 'issue': 'missing_in_ledger',
                  'ledger_amount': np.nan, 'gateway_amount': r['amount_usd'],
                  'ledger_status': np.nan, 'gateway_status': r['status']})

amount_txn_ids = set(amount_mismatches['transaction_id'])
for _, r in amount_mismatches.iterrows():
    recon.append({'transaction_id': r['transaction_id'], 'issue': 'amount_mismatch',
                  'ledger_amount': r['amount_usd_ledger'], 'gateway_amount': r['amount_usd_gateway'],
                  'ledger_status': r['status_ledger'], 'gateway_status': r['status_gateway']})

for _, r in status_mismatches.iterrows():
    if r['transaction_id'] not in amount_txn_ids:
        recon.append({'transaction_id': r['transaction_id'], 'issue': 'status_mismatch',
                      'ledger_amount': r['amount_usd_ledger'], 'gateway_amount': r['amount_usd_gateway'],
                      'ledger_status': r['status_ledger'], 'gateway_status': r['status_gateway']})

recon_df = pd.DataFrame(recon)
print(f"Total reconciliation issues: {len(recon_df)}")
print(recon_df.to_string(index=False))

recon_df.to_csv(PROCESSED / 'reconciliation_report.csv', index=False)
print("\nSaved: reconciliation_report.csv")


Total reconciliation issues: 6
 transaction_id              issue  ledger_amount  gateway_amount ledger_status gateway_status
           R004  missing_in_gateway         2100.0             NaN       success            NaN
           R010  missing_in_gateway         2500.0             NaN       success            NaN
           R011   missing_in_ledger            NaN          1800.0           NaN        success
           R002      amount_mismatch          850.0           900.0       success        success
           R008      amount_mismatch          640.0           600.0       success        success
           R005       status_mismatch         7200.0          7200.0       success         failed

Saved: reconciliation_report.csv


### Step 8 — Summary Metrics JSON

In [ ]:
amount_at_risk = round(
    missing_in_gateway['amount_usd'].sum() +
    amount_mismatches['amount_diff'].abs().sum(), 2
)

metrics = {
    "total_ledger_rows": len(ledger),
    "total_gateway_rows": len(gateway),
    "missing_in_gateway_count": len(missing_in_gateway),
    "missing_in_ledger_count": len(missing_in_ledger),
    "amount_mismatch_count": len(amount_mismatches),
    "status_mismatch_count": len(status_mismatches),
    "reconciliation_issue_count": len(recon_df),
    "ledger_total_amount": round(ledger['amount_usd'].sum(), 2),
    "gateway_total_amount": round(gateway['amount_usd'].sum(), 2),
    "amount_at_risk": amount_at_risk
}

print(json.dumps(metrics, indent=2))

with open('../04_python/summary_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("\nSaved: summary_metrics.json")


{
  "total_ledger_rows": 10,
  "total_gateway_rows": 9,
  "missing_in_gateway_count": 2,
  "missing_in_ledger_count": 1,
  "amount_mismatch_count": 2,
  "status_mismatch_count": 1,
  "reconciliation_issue_count": 6,
  "ledger_total_amount": 23340.0,
  "gateway_total_amount": 20550.0,
  "amount_at_risk": 4690.0
}

Saved: summary_metrics.json


---
## Part 4: JSON Normalization

Flatten the nested `api_response_sample.json` from QuickPay's Settlement API into a flat tabular format.


### Step 1 — Load and Inspect the JSON

In [ ]:
with open(RAW / 'api_response_sample.json') as f:
    api = json.load(f)

print("Top-level keys:", list(api.keys()))
print("generated_at:", api['generated_at'])
print("source:", api['source'])
print("Number of batches:", len(api['batches']))
print("\nBatch 0 keys:", list(api['batches'][0].keys()))
print("Settlement keys:", list(api['batches'][0]['settlements'][0].keys()))


Top-level keys: ['generated_at', 'source', 'batches']
generated_at: 2026-03-07T10:00:00Z
source: QuickPay Settlement API
Number of batches: 2

Batch 0 keys: ['batch_id', 'merchant', 'settlements']
Settlement keys: ['settlement_id', 'amount_usd', 'status', 'processed_at', 'bank']


### Step 2 — Flatten Nested Structure

In [ ]:
rows = []
for batch in api['batches']:
    for s in batch['settlements']:
        rows.append({
            'generated_at':    api['generated_at'],
            'source':          api['source'],
            'batch_id':        batch['batch_id'],
            'merchant_id':     batch['merchant']['merchant_id'],
            'merchant_name':   batch['merchant']['merchant_name'],
            'merchant_region': batch['merchant']['region'],
            'settlement_id':   s['settlement_id'],
            'amount_usd':      s['amount_usd'],
            'status':          s['status'],
            'processed_at':    s['processed_at'],
            'bank_name':       s['bank']['name'],
            'bank_country':    s['bank']['country'],
        })

api_df = pd.DataFrame(rows)
print("Flattened shape:", api_df.shape)
print("\nColumn names:", api_df.columns.tolist())


Flattened shape: (6, 12)

Column names: ['generated_at', 'source', 'batch_id', 'merchant_id', 'merchant_name', 'merchant_region', 'settlement_id', 'amount_usd', 'status', 'processed_at', 'bank_name', 'bank_country']


### Step 3 — Clean Column Names and Convert Datetime Fields

In [ ]:
# Column names are already snake_case — no cleanup needed
# Convert datetime strings to pandas datetime
api_df['generated_at'] = pd.to_datetime(api_df['generated_at'], utc=True)
api_df['processed_at'] = pd.to_datetime(api_df['processed_at'], utc=True)

print("Dtypes after conversion:")
print(api_df.dtypes.to_string())
print()
print(api_df.to_string(index=False))


Dtypes after conversion:
generated_at      datetime64[ns, UTC]
source                          object
batch_id                        object
merchant_id                     object
merchant_name                   object
merchant_region                 object
settlement_id                   object
amount_usd                     float64
status                          object
processed_at      datetime64[ns, UTC]
bank_name                       object
bank_country                    object
dtype: object

              generated_at                   source batch_id merchant_id  merchant_name merchant_region settlement_id  amount_usd   status               processed_at bank_name bank_country
2026-03-07 10:00:00+00:00  QuickPay Settlement API     B001        M001     Alpha Mart            APAC          S001      1520.5  settled  2026-03-07 08:10:00+00:00    Bank A           IN
2026-03-07 10:00:00+00:00  QuickPay Settlement API     B001        M001     Alpha Mart            APAC          S002 

### Step 4 — Save Normalized Output

In [ ]:
api_df.to_csv(PROCESSED / 'api_normalized.csv', index=False)
print("Saved: api_normalized.csv")
print(f"Shape: {api_df.shape}")
print(f"Total settlement amount: ${api_df['amount_usd'].sum():,.2f}")
print(f"Settled: {(api_df['status']=='settled').sum()} | Pending: {(api_df['status']=='pending').sum()} | Failed: {(api_df['status']=='failed').sum()}")


Saved: api_normalized.csv
Shape: (6, 12)
Total settlement amount: $12,940.50
Settled: 4 | Pending: 1 | Failed: 1


---
## Dashboard Source Outputs

Generate the 4 aggregated CSVs needed to power the Looker Studio dashboard.


### Load Cleaned Transactions

In [ ]:
tx = pd.read_csv(PROCESSED / 'cleaned_transactions.csv')
print("Cleaned transactions shape:", tx.shape)
print("Date range:", tx['transaction_date'].min(), "to", tx['transaction_date'].max())


Cleaned transactions shape: (30, 15)
Date range: 2026-03-01 to 2026-03-06


### Daily Summary

In [ ]:
daily = tx.groupby('transaction_date').agg(
    total_transactions=('transaction_id', 'count'),
    total_gmv_usd=('amount_usd', 'sum'),
    captured_gmv_usd=('amount_usd', lambda x: x[tx.loc[x.index, 'status'] == 'captured'].sum()),
    success_count=('status', lambda x: (x == 'captured').sum()),
    failed_count=('status', lambda x: (x == 'failed').sum()),
    chargeback_count=('status', lambda x: (x == 'chargeback').sum()),
).reset_index()
daily['success_rate_pct'] = round(daily['success_count'] / daily['total_transactions'] * 100, 2)
daily['total_gmv_usd'] = daily['total_gmv_usd'].round(2)
daily['captured_gmv_usd'] = daily['captured_gmv_usd'].round(2)

daily.to_csv(PROCESSED / 'daily_summary.csv', index=False)
print("Saved: daily_summary.csv")
print(daily.to_string(index=False))


Saved: daily_summary.csv
 transaction_date  total_transactions  total_gmv_usd  captured_gmv_usd  success_count  failed_count  chargeback_count  success_rate_pct
       2026-03-01                   5        26382.0           26382.0              5             0                 0            100.00
       2026-03-02                   6        25049.0           11080.0              3             1                 2             50.00
       2026-03-03                   5        18391.0           16031.5              4             1                 0             80.00
       2026-03-04                   5        16420.0           13920.0              4             0                 1             80.00
       2026-03-05                   6        19232.0            6136.0              1             4                 1             16.67
       2026-03-06                   3        10606.0            8806.0              2             1                 0             66.67


### Payment Method Breakdown

In [ ]:
pm = tx.groupby('payment_method').agg(
    total_transactions=('transaction_id', 'count'),
    total_gmv_usd=('amount_usd', 'sum'),
    captured_gmv_usd=('amount_usd', lambda x: x[tx.loc[x.index, 'status'] == 'captured'].sum()),
).reset_index()
pm['total_gmv_usd'] = pm['total_gmv_usd'].round(2)
pm['captured_gmv_usd'] = pm['captured_gmv_usd'].round(2)
pm.to_csv(PROCESSED / 'payment_method_breakdown.csv', index=False)
print("Saved: payment_method_breakdown.csv")
print(pm.to_string(index=False))


Saved: payment_method_breakdown.csv
   payment_method  total_transactions  total_gmv_usd  captured_gmv_usd
           Card                  13        52972.0           34283.0
     NetBanking                   3        15306.0           11709.0
            UPI                   9        34397.5           28527.0
         Wallet                   5        13404.5            7836.5


### Region Breakdown

In [ ]:
rb = tx.groupby('gateway_region').agg(
    total_transactions=('transaction_id', 'count'),
    total_gmv_usd=('amount_usd', 'sum'),
    captured_gmv_usd=('amount_usd', lambda x: x[tx.loc[x.index, 'status'] == 'captured'].sum()),
    avg_risk_score=('risk_score', 'mean'),
    chargeback_count=('status', lambda x: (x == 'chargeback').sum()),
    high_risk_count=('high_risk_flag', 'sum'),
).reset_index()
rb['avg_risk_score'] = rb['avg_risk_score'].round(2)
rb['total_gmv_usd'] = rb['total_gmv_usd'].round(2)
rb['captured_gmv_usd'] = rb['captured_gmv_usd'].round(2)
rb.to_csv(PROCESSED / 'region_breakdown.csv', index=False)
print("Saved: region_breakdown.csv")
print(rb.to_string(index=False))


Saved: region_breakdown.csv
 gateway_region  total_transactions  total_gmv_usd  captured_gmv_usd  avg_risk_score  chargeback_count  high_risk_count
           APAC                  22        82594.0           63415.5           65.48                 2                7
             EU                   4        18886.0            8640.0           47.25                 1                1
             US                   4        14600.0           10300.0           48.75                 1                1


### Merchant Performance Summary

In [ ]:
mp = tx.groupby(['merchant_name', 'merchant_id', 'merchant_category', 'gateway_region']).agg(
    total_transactions=('transaction_id', 'count'),
    total_gmv_usd=('amount_usd', 'sum'),
    captured_gmv_usd=('amount_usd', lambda x: x[tx.loc[x.index, 'status'] == 'captured'].sum()),
    failed_count=('status', lambda x: (x == 'failed').sum()),
    chargeback_count=('status', lambda x: (x == 'chargeback').sum()),
    avg_risk_score=('risk_score', 'mean'),
    high_risk_count=('high_risk_flag', 'sum'),
    high_value_count=('high_value_flag', 'sum'),
).reset_index()
mp['chargeback_ratio_pct'] = round(mp['chargeback_count'] / mp['total_transactions'] * 100, 2)
mp['success_rate_pct'] = round((mp['total_transactions'] - mp['failed_count'] - mp['chargeback_count']) / mp['total_transactions'] * 100, 2)
mp['avg_risk_score'] = mp['avg_risk_score'].round(2)
mp['total_gmv_usd'] = mp['total_gmv_usd'].round(2)
mp['captured_gmv_usd'] = mp['captured_gmv_usd'].round(2)
mp.to_csv(PROCESSED / 'merchant_performance_summary.csv', index=False)
print("Saved: merchant_performance_summary.csv")
print(mp.to_string(index=False))


Saved: merchant_performance_summary.csv
   merchant_name merchant_id merchant_category gateway_region  total_transactions  total_gmv_usd  captured_gmv_usd  failed_count  chargeback_count  avg_risk_score  high_risk_count  high_value_count  chargeback_ratio_pct  success_rate_pct
    Alpha Mart        M001           Grocery           APAC                  11        40812.0           29984.5             2                 1           61.20                2                 2                  9.09             72.73
   Beta Stores        M002       Electronics           APAC                  11        41782.0           33431.0             3                 1           69.36                5                 3                  9.09             63.64
   City Pharma        M003        Healthcare             EU                   2         8640.0            8640.0             0                 0           40.00                0                 0                  0.00            100.00
 Delta Travels

---
## Summary

| Output File | Rows | Description |
|---|---|---|
| missing_in_gateway.csv | 2 | Ledger records not found in gateway |
| missing_in_ledger.csv | 1 | Gateway records not found in ledger |
| amount_mismatches.csv | 2 | Same transaction_id, different amount |
| status_mismatches.csv | 1 | Same transaction_id, different status |
| reconciliation_report.csv | 6 | All issues consolidated |
| api_normalized.csv | 6 | Flattened JSON settlements |
| daily_summary.csv | 6 | Daily GMV and performance |
| payment_method_breakdown.csv | 4 | GMV by payment method |
| region_breakdown.csv | 3 | GMV and risk by region |
| merchant_performance_summary.csv | 5 | Full merchant metrics |
| summary_metrics.json | — | 10 key reconciliation metrics |

**Amount at Risk: $4,690.00** (missing in gateway $4,600 + amount diff $90)
